# Módulo 2: INSERT INTO, INSERT OVERWRITE, hash y MERGE


## 2.1: INSERT INTO — Carga incremental


In [0]:
%sql

drop table if exists propiedades.silver.propiedades_into;

CREATE TABLE propiedades.silver.propiedades_into
(
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP
)
USING DELTA
TBLPROPERTIES
(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Tabla de prueba de Insert INTO';

In [0]:
%sql
-- Primera carga: alquileres

INSERT INTO propiedades.silver.propiedades_into

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM propiedades.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 500;

In [0]:
%sql
-- Segunda carga: ventas (INSERT INTO acumula)

INSERT INTO propiedades.silver.propiedades_into

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM propiedades.silver.propiedades

WHERE tipo_operacion = 'venta'
LIMIT 500;

SELECT  tipo_operacion, COUNT(*) as cantidad
FROM    propiedades.silver.propiedades_into
GROUP BY tipo_operacion;

## 2.2: INSERT INTO — Prevenir duplicados


In [0]:
%sql
INSERT INTO propiedades.silver.propiedades_into

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM propiedades.silver.propiedades sp
WHERE tipo_operacion = 'alquiler_temporario'
    AND NOT EXISTS (
        SELECT 1 
        FROM propiedades.silver.propiedades_into pi
        WHERE pi.url = sp.url
          AND pi.precio = sp.precio                 -- Lo unimos por las llaves de la tabla
    )                                               -- Validamos que no Exista en la tabla
LIMIT 300;

SELECT COUNT(*) as total_registros FROM propiedades.silver.propiedades_into;

## 2.3: INSERT OVERWRITE — Reprocesamiento


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.propiedades_overwrite;

CREATE TABLE propiedades.silver.propiedades_overwrite 
(
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para INSERT OVERWRITE';


In [0]:
%sql

INSERT OVERWRITE propiedades.silver.propiedades_overwrite

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM propiedades.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 1000;


SELECT COUNT(*) as total_registros,
        tipo_operacion
FROM propiedades.silver.propiedades_overwrite
GROUP by tipo_operacion;

-- Segunda carga: OVERWRITE reemplaza todo
-- Cargamos set: VENTA

In [0]:
%sql

INSERT OVERWRITE propiedades.silver.propiedades_overwrite

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM propiedades.silver.propiedades
WHERE tipo_operacion = 'venta'
LIMIT 1000;


SELECT COUNT(*) as total_registros,
        tipo_operacion
FROM propiedades.silver.propiedades_overwrite
GROUP by tipo_operacion;

## 2.4: Generar hash con MD5


In [0]:
%sql
-- La variacion de 1 letra provoca un cambio total de la cadena

SELECT 
    MD5('hola') as hash_simple,
    MD5('hola') as hash_repetido,
    MD5('Hola') as hash_diferente;      -- Mayuscula letra 1

In [0]:
%sql
SELECT  MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
        *
                
FROM propiedades.silver.propiedades
LIMIT 5;

## 2.5: MERGE — Solo INSERT


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.propiedades_merge_insert;

CREATE TABLE propiedades.silver.propiedades_merge_insert (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING PRIMARY KEY,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES
(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Tabla de prueba para MERGE (solo INSERT)'
;


In [0]:
%sql

INSERT INTO propiedades.silver.propiedades_merge_insert 

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM    propiedades.silver.propiedades
WHERE   tipo_operacion = 'alquiler'
LIMIT 100;


SELECT  COUNT(*) as total_inicia,
        tipo_operacion
FROM    propiedades.silver.propiedades_merge_insert
GROUP BY tipo_operacion;

%md
MERGE INTO Tabla_Target

(
    SELECT FUENTE
)

ON ID = ID

WHEN NOT MATCHED THEN
    INSERT INTO

WHEN MATCHED THEN
    UPDATE SET

In [0]:
%sql
MERGE INTO propiedades.silver.propiedades_merge_insert AS TARGET
USING
(
        SELECT  partido, region, tipo_operacion, precio, moneda,
                metros_cuadrados_totales, precio_por_m2, url

        FROM bootcamp.silver.propiedades
        WHERE tipo_operacion IN ('alquiler', 'venta')
        LIMIT 200
) as source
ON target.url = source.url

WHEN NOT MATCHED THEN 
    ---> Columnas de la tabla Fuente
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)

    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url, CURRENT_TIMESTAMP());




------------------------------
-- Validamos los registros insertados
------------------------------
SELECT  tipo_operacion, 
        COUNT(*) as cantidad
FROM    propiedades.silver.propiedades_merge_insert
GROUP BY tipo_operacion;

##  2.6: MERGE — UPSERT
-- AGregamos el WHen Matched then 

In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.propiedades_merge_upsert;


CREATE TABLE propiedades.silver.propiedades_merge_upsert 
(
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE (UPSERT)';


-------------------------------------
-- Insertamos registros iniciales
-------------------------------------
INSERT INTO propiedades.silver.propiedades_merge_upsert

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp,
        CURRENT_TIMESTAMP() as _updated_at

FROM    propiedades.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;


SELECT  COUNT(*) as total_inicial 
FROM    propiedades.silver.propiedades_merge_upsert;

In [0]:
%sql
--------------------------------------
-- creamos una vista para usarla como fuente en el merge
--------------------------------------

CREATE OR REPLACE TEMP VIEW datos_actualizados_vw 
AS
SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url
FROM (
    SELECT  partido, region, tipo_operacion,
            precio * 1.10 as precio, moneda,
            metros_cuadrados_totales,
            precio_por_m2 * 1.10 as precio_por_m2, url,
            ROW_NUMBER() OVER (PARTITION BY precio, url ORDER BY precio) as rn

    FROM propiedades.silver.propiedades
    WHERE tipo_operacion = 'alquiler'
) a
WHERE rn = 1
LIMIT 150;

--------------------------------------
-- Merge whith MATCHED (update)
--------------------------------------

MERGE INTO propiedades.silver.propiedades_merge_upsert AS target
USING datos_actualizados_vw AS source

ON  target.url = source.url 
AND target.precio = source.precio

WHEN MATCHED THEN
    UPDATE SET
        precio_por_m2   = source.precio_por_m2,
        _updated_at     = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
            metros_cuadrados_totales, precio_por_m2, url,
            _processing_timestamp, _updated_at)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
            source.metros_cuadrados_totales, source.precio_por_m2, source.url,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());


--------------------------------------------
--Validamos
--------------------------------------------
SELECT COUNT(*) as total_despues_upsert
FROM propiedades.silver.propiedades_merge_upsert; 

## 2.7: MERGE con Hash — Deduplicación simplificada


In [0]:
%sql

----------------------------------
-- Carga Inicial
----------------------------------
DROP TABLE IF EXISTS propiedades.silver.propiedades_merge_hash;

CREATE TABLE propiedades.silver.propiedades_merge_hash
(
    row_hash STRING NOT NULL,
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con Hash';


-----------------------------------------
-- creamos nueva columna HASH con MD5
-----------------------------------------
INSERT INTO propiedades.silver.propiedades_merge_hash

SELECT 
        MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
        partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP() as _processing_timestamp

FROM    propiedades.silver.propiedades
WHERE   tipo_operacion = 'alquiler'
LIMIT 200;


-----------------------------------------
-- validamos los registros 
-----------------------------------------
SELECT  COUNT(*) as total_inicial 
FROM    propiedades.silver.propiedades_merge_hash;

In [0]:
%sql

MERGE INTO propiedades.silver.propiedades_merge_hash AS target

USING (
    SELECT 
            MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
            partido, region, tipo_operacion, precio, moneda,
            metros_cuadrados_totales, precio_por_m2, url
    FROM propiedades.silver.propiedades
    WHERE tipo_operacion IN ('alquiler', 'venta')
    LIMIT 400
) AS source

ON target.row_hash = source.row_hash        -- Usamos 1 sola columna como llave MD5 (concatena 3 columnas)

WHEN NOT MATCHED THEN
    INSERT (row_hash, partido, region, tipo_operacion, precio, moneda,
            metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)
    VALUES (source.row_hash, source.partido, source.region, source.tipo_operacion, source.precio,
            source.moneda, source.metros_cuadrados_totales, source.precio_por_m2,
            source.url, CURRENT_TIMESTAMP()
            );

SELECT  COUNT(*) as total_despues_merge, 
        COUNT(DISTINCT row_hash) as hashes_unicos
FROM    propiedades.silver.propiedades_merge_hash;

##2.8: MERGE — UPSERT + DELETE (soft delete)


In [0]:
%sql

----------------------------------------------------------------
-- creamos tabla
----------------------------------------------------------------
DROP TABLE IF EXISTS propiedades.silver.propiedades_merge_delete;

CREATE TABLE propiedades.silver.propiedades_merge_delete 
(
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _is_deleted BOOLEAN DEFAULT false
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con soft delete';


----------------------------------------------------------------
-- insertamos datos iniciales
----------------------------------------------------------------
INSERT INTO propiedades.silver.propiedades_merge_delete

SELECT  partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false
        
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

------------------------------
-- WHEN NOT MATCHED BY SOURCE 
------------------------------

cuando existe un registro en la tabla destino (target) pero ya no existe en la tabla origen (source).
------------------------------

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW datos_nuevos_delete AS
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 50;

MERGE INTO propiedades.silver.propiedades_merge_delete AS target

USING datos_nuevos_delete AS source

ON target.url = source.url

WHEN MATCHED THEN
    UPDATE SET
        precio = source.precio,
        precio_por_m2 = source.precio_por_m2,
        _updated_at = CURRENT_TIMESTAMP(),
        _is_deleted = false

WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        _processing_timestamp, _updated_at, _is_deleted)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false)

------------------------------
-- cuando existe un registro en la tabla destino (target) pero ya no existe en la tabla origen (source).
---------------------------------
WHEN NOT MATCHED BY SOURCE THEN
    UPDATE SET _is_deleted = true, _updated_at = CURRENT_TIMESTAMP();



SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN _is_deleted THEN 1 ELSE 0 END) as soft_deleted,
    SUM(CASE WHEN NOT _is_deleted THEN 1 ELSE 0 END) as activos
FROM bootcamp.silver.propiedades_merge_delete;